# Getting Started

![pyku Logo](../_static/pyku_logo.jpg)

[pyku](http://ku.pages.dwd.de/libraries/pyku) is an extension built on xarray for **handling climate data comprehensively**. Its primary aim is to facilitate the reading and manipulation of all climate-related data and metadata. *pyku* enables tasks such as resampling datetimes, applying masks, polygonizing, reprojection, basic analysis, CMORization, and validation. It is **designed to foster collaborative development**, emphasizing best practices in code review, style adherence, and deployment.

*pyku* **standardizes our codebase** concerning technicalities such as installation and management of dependencies, and resources such as geographic projections, polygons, and color scales.

## Collaborative Development

http://ku.pages.dwd.de/documentation/starter/quality.html

* Version Management
* Code Review
* Code Documentation
* Programming style guideline
* Unit testing
* Integration Testing
* CI/CD Pipelines

## Core Features

* Handles all tasks associated with geographic projections
* Parses and manages all climate-related metadata
* Converts polygons and points into raster format
* Converts rasters into polygon data
* Generates masks from files, creates masks from polygons, and applies them
* Visualizes fundamental quantities through plotting
* Automatically converts units
* Conducts data and metadata integrity checks
* Automatically resamples datetime and sets attributes
* Dynamically fixes known issues in datasets
* Applies CMORization to any dataset
* Executes bias correction procedures
* Performs downscaling operations

## The Mighty Xarray Library

> Xarray makes working with labelled multi-dimensional arrays in Python simple, efficient, and fun!

Ideally, we should **avoid writing or re-writing code** if it is already available and maintained. For this purpose, *pyku* is fully integrated with Xarray, extending its datasets with functions focused on handling climate data, particularly the data we use at KU. Since *pyku* is built within the Xarray framework, it is **fully compatible with a large number of high-quality Xarray extensions**, allowing us to use them without needing to write and maintain them ourselves. Here are a few examples:

**MetPy**

> MetPy is a collection of tools in Python for reading, visualizing, and performing calculations with weather data.

**xclim**

> xclim is an operational Python library for climate services, providing numerous climate-related indicator tools with an extensible framework for constructing custom climate indicators, statistical downscaling and bias adjustment of climate model simulations, as well as climate model ensemble analysis tools.

**xskillscore**

> Metrics for verifying forecasts

**Satpy**

> Satpy is a python library for reading, manipulating, and writing data from remote-sensing earth-observing satellite instruments.

</br>

We load the Xarray and *pyku* libraries, along with the ``analyze`` module from *pyku*, to perform some plotting during this walk-through.

In [ ]:
import xarray as xr
import pyku
import pyku.analyse as analyse

</br>

## Selecting Real-world Data for this Walk-through

First, we define real-world test data from the `pyku.resources` module.

In [ ]:
pyku.list_test_data()

</br>

## Checking and Repairing Broken Datasets

This is generally where we begin our analysis: by going through individual datasets. *pyku* not only allows us to **check datasets for common issues** but also **tentatively repair on-the-fly** any datasets that we use at KU which may be broken.

In [ ]:
ds = pyku.resources.get_test_data('hyras_pr')

</br>

As previously mentioned, *pyku* is built on xarray and leverages its **Dataset accessor to seamlessly integrate additional features**. These features can be accessed using the *pyku* accessor, as demonstrated below with the use of the check functions.


In [ ]:
ds.pyku.check_metadata().query("issue.notna()")

</br>

In the HYRAS data mentioned above, we have identified several issues that can be repaired on-the-fly. Before proceeding, note that you can access the documentation directly in IPython by using the `?` command.

In [ ]:
ds.pyku.check_metadata?

</br>

To address the issues identified, you can utilize the *pyku* standard functions as well as xarray standard functionalities

In [ ]:
# Convert to cmor units
ds = ds.pyku.to_cmor_units()

# Set the time label at 00:00 every day
ds = ds.assign_coords(time=ds.time.dt.floor('1D'))

In [ ]:
ds.pyku.check_metadata()

</br>

Now we check the data again to make sure that the repair solved the issues.

In [ ]:
ds.pyku.check_metadata().query("issue.notna()")

</br>

## Processing Datasets On-the-Fly During Opening

Xarray allows passing a function to process all files as they are opened, making it an ideal stage for preprocessing! Common tasks include:

- Adapting data to your needs
- Setting time labels
- Standardizing variable names
- Standardizing units

To achieve this, we define a function that accepts a dataset, performs operations on it, and returns the modified dataset.

In [ ]:
def preprocess(ds):

    if 'number_of_stations' in ds.data_vars:
        ds = ds.drop_vars('number_of_stations')

    ds = ds.pyku.to_cmor_units()
    ds = ds.pyku.to_cmor_varnames()
    ds = ds.assign_coords(time=ds.time.dt.floor('1D'))
    ds = ds.pyku.wrap_longitudes()
    ds = ds.sel(time=slice('1981', '1982'))
    
    return ds

</br>

This function can be applied during the file opening process!

**Note:** *pyku* offers functions for preprocessing data into a dedicated scratch directory, optionally using parallelism. This functionality is useful for handling large datasets efficiently.

In [ ]:
# Open the model files
model = pyku.resources.get_test_data('MPI_ESM1_2_HR_pr')

# Open the reference HYRAS files
reference = pyku.resources.get_test_data('hyras_pr')

In [ ]:
model.pyku.check_metadata().query("issue.notna()")

In [ ]:
reference.pyku.check_metadata().query("issue.notna()")

In [ ]:
model = preprocess(model)
reference = preprocess(reference)

</br>

## Comparing Datasets

At this stage, it's beneficial to compare datasets for fundamental differences. For instance, we may observe that the datasets do not share the same geographic projection.

In [ ]:
model.pyku.compare_datasets(reference)

</br>

We have a look at the first timestep in the data for a quick sanity check. We spot that the geographic projection used, as well as the masking is not the same.

In [ ]:
# Set labels
model = model.assign_attrs({'label': 'Model'})
reference = reference.assign_attrs({'label': 'Reference'})

analyse.n_maps(
    model.isel(time=0),
    reference.isel(time=0),
    var='pr'
)

</br>

## Managing Geographic Projections

*pyku* enables robust handling of geographic projections, which is a core feature of the tool. It supports a wide range of projections, emphasizing standardization across datasets. *pyku* can manage and write all projection metadata (PROJ string, WKT, CF-conform standard). Refining these standard projection definitions is a continuous and collaborative work.

For a comprehensive list of standardized areas, refer to the documentation or access them directly through *pyku*.

In [ ]:
pyku.list_standard_areas()

</br>

We can project our data onto the HYRAS grid at a 5-kilometer resolution or onto any of the standardized projections listed above.

**Note**: The HYRAS data are typically already in the projection. Occasionally, the last digits of the georeferencing is not exactly the same depending on what code was used. For simplicity, we reproject the HYRAS data here. The *compare* functions in *pyku* will help us determine if the georeferencing alignment is compatible (a check for exact georeferencing alignment will be included in the future).

The code would look like:

````
projected_model = model.pyku.project('HYR-LAEA-5')
projected_model.pyku.align_georeferencing(reference)
````

In [ ]:
# Project to the HYRAS grid
projected_model = model.pyku.project('HYR-LAEA-5')
projected_reference = reference.pyku.project('HYR-LAEA-5')

In [ ]:
# Set labels
projected_model = projected_model.assign_attrs({'label': 'Projected model'})
projected_reference = projected_reference.assign_attrs({'label': 'Projected reference'})

analyse.n_maps(
    projected_model.isel(time=0),
    projected_reference.isel(time=0),
    var='pr',
    crs='HYR-LAEA-5'
)

</br>

Note that during that operation, all geographic projection metadata have been set automatically!

Note the grid mapping attribute, as well as the crs variable

In [ ]:
projected_model.pr.attrs

In [ ]:
# Note the CRS attributes
projected_model.crs.attrs

</br>

## Masking and Polygon Resources

A common basic operation involves selecting a region and masking the data. *pyku* facilitates the standardization and application of polygons for this purpose.

**Note**: Functions are available to directly obtain mask polygons from a dataset, generate masks in Xarray format, create masks for multiple polygons, or segment datasets into regions. For detailed instructions, please refer to the documentation as this is a brief overview.

First we import the *pyku* ``resources`` module and show the available polygons.

In [ ]:
import pyku.resources as resources

In [ ]:
resources.get_polygon_identifiers()

</br>

Here we will select Germany, which is returned as a GeoDataFrame downloaded on-the-fly and on-demand from the original source.

In [ ]:
germany = resources.get_geodataframe('germany')
germany

</br>

And now we can apply the mask directly to our datasets.

In [ ]:
masked_model = projected_model.pyku.apply_mask(germany)
masked_reference = projected_reference.pyku.apply_mask(germany)

</br>
A quick sanity check now shows us the projected and masked datasets

In [ ]:
# Set labels
masked_model = masked_model.assign_attrs({'label': 'Projected model'})
masked_reference = masked_reference.assign_attrs({'label': 'Projected reference'})

analyse.n_maps(
    masked_model.isel(time=0),
    masked_reference.isel(time=0),
    var='pr',
    crs='HYR-LAEA-5'
)

</br>

## Basic Plotting

*pyku* features an `analyse` module designed for simple plotting and dataset exploration. While currently intended for basic visualization rather than comprehensive analysis, future developments may expand its capabilities in that direction. Given the small size of the dataset used here, it can be fully loaded into memory for optimal performance in this notebook. However, *pyku* can handle datasets of any size without limitations.

In [ ]:
masked_model = masked_model.compute()
masked_reference = masked_reference.compute()

</br>

And here we can have a look for example at the monthly probability distribution function. We set the label of the data and plot.

In [ ]:
# Set labels
masked_model = masked_model.assign_attrs({'label': 'Model'})
masked_reference = masked_reference.assign_attrs({'label': 'Reference'})

analyse.monthly_pdf(
    masked_model,
    masked_reference,
    var='pr',
    nbins=100,
    yscale='log'
)

</br>

## Resampling Datetimes

The functionnality for resampling datetimes is certainly of generic interest. First, let's examine the dataset and isolate the temperature data. Note that the dataset includes `time_bounds` and `crs` as data variables.

In [ ]:
masked_model.data_vars

</br>
If we were to select the temperature using the standard Xarray functions, we would lose that information.

In [ ]:
masked_model[['pr']].data_vars

</br>

*pyku* offers an alternative selection method to obtain a dataset with all the relevant climate data information.

In [ ]:
pr_data = masked_model.pyku.get_geodataset('pr')
pr_data.data_vars

</br>

We can now resample the data to a monthly frequency, specifying the resampling method (here, using the sum). The time bounds have been automatically adjusted during resampling.

In [ ]:
pr_data.pyku.resample_datetimes(how='sum', frequency='1MS')

</br>

## The KU Colormap

*pyku* includes the KU standard colormap for consistent use across our projects. Implementing this colormap aims to streamline our workflow by eliminating the need to define the colormap in each code individually. By importing the `colormap` module, updating to the latest version via `pip install --upgrade` ensures all codes are synchronized.

Let's begin by importing the module and visualizing all available colormaps.

In [ ]:
import pyku.colormaps as colormaps

In [ ]:
colormaps.plot_colormaps(kind='original')

</br>

Once we've selected a colormap of interest, we can apply it using the `cmap` keyword. In Python, `cmap` from Matplotlib serves as the de facto standard for colormaps, compatible with virtually any plotting library.

In [ ]:
mycolormap = colormaps.get_cmap('temp_ano', kind='original')

</br>

For example, the plotting routines in the ``analyse`` module can take any relevant matplotlib keywords, and in particular the ``cmap`` keyword.

In [ ]:
# Set labels
masked_model = masked_model.assign_attrs({'label': 'Projected model'})
masked_reference = masked_reference.assign_attrs({'label': 'Projected reference'})

analyse.n_maps(
    masked_model.isel(time=10),
    masked_reference.isel(time=0),
    cmap=mycolormap,
    var='pr',
    crs='HYR-LAEA-5'
)